In [1]:
!pip install ultralytics==8.4.7 markdown rich wrapt pandas huggingface_hub opencv-python wandb datasets librosa -q

In [2]:
from datasets import load_dataset
import ultralytics
import os
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset, DatasetDict, ClassLabel, Audio as DatasetAudio
from PIL import Image
import shutil
from tqdm.auto import tqdm
from ultralytics import YOLO
import yaml
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
import matplotlib.pyplot as plt
import librosa.display

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ultralytics.checks()

Ultralytics 8.4.7 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA A10, 22588MiB)
Setup complete ✅ (96 CPUs, 377.7 GB RAM, 4814.5/5037.1 GB disk)


In [4]:
SAMPLING_RATE = 16000
LABELS = ['other', 'drone']

# Paths
SPECTROGRAM_DIR = Path("./spectrograms")
TRAIN_DIR = SPECTROGRAM_DIR / "train"
VAL_DIR = SPECTROGRAM_DIR / "val"
TEST_DIR = SPECTROGRAM_DIR / "test"
NUM_WORKERS = 24


In [5]:
# Load dataset
print("Loading dataset...")
dataset = load_dataset("Hibou-Foundation/big_ds_4_raw_wav_balanced")
print(f"\nInitial dataset structure: {dataset}")
print(f"Initial features: {dataset['train'].features}")
dataset = dataset.cast_column("label", ClassLabel(names=LABELS))

sample_item = dataset['train'][0]
print(f"\nSample item keys: {sample_item.keys()}")
print(f"Audio type: {type(sample_item['audio'])}")

# Take only n percent if the dataset
dataset = DatasetDict({
    split: ds.select(range(int(0.01 * len(ds))))
    for split, ds in dataset.items()
})
print({k: v.shape for k, v in dataset.items()})


Loading dataset...

Initial dataset structure: DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 352116
    })
    val: Dataset({
        features: ['audio', 'label'],
        num_rows: 43580
    })
    test: Dataset({
        features: ['audio', 'label'],
        num_rows: 43478
    })
})
Initial features: {'audio': List(Value('float32')), 'label': ClassLabel(names=['other', 'drone'])}

Sample item keys: dict_keys(['audio', 'label'])
Audio type: <class 'list'>
{'train': (3521, 2), 'val': (435, 2), 'test': (434, 2)}


In [6]:
def convert_to_linear_spectrogram(batch):
    all_linear_db = []
    all_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        data = np.array(audio)
        # Compute STFT
        stft = librosa.stft(data, n_fft=2048, hop_length=256)

        # Compute magnitude
        magnitude = np.abs(stft)

        # Convert to dB
        linear_db = librosa.amplitude_to_db(magnitude, ref=np.max)
        all_linear_db.append(torch.tensor(linear_db))
        all_labels.append(torch.tensor(label))

    return {
        # Convert to torch tensors
        "audio": all_linear_db,
        "label": all_labels,
    }



spectrogram_dataset = DatasetDict()
for split in dataset.keys():
    print(f"Converting to spectrogram split: {split}")
    spectrogram_split = dataset[split].map(
        convert_to_linear_spectrogram,
        batched=True,
        num_proc=20,
        batch_size=32,
        remove_columns=dataset[split].column_names,
    )
    spectrogram_dataset[split] = spectrogram_split

Converting to spectrogram split: train
Converting to spectrogram split: val
Converting to spectrogram split: test


In [7]:
# Create directory structure for YOLO classification format
def create_yolo_structure():
    for split in ['train', 'val', 'test']:
        for _label in LABELS:
            (SPECTROGRAM_DIR / split / _label).mkdir(parents=True, exist_ok=True)

create_yolo_structure()


In [8]:
def save_spectogram_to_file(spectrogram, path, width=480, height=480, cmap='viridis'):
    spec_array = np.array(spectrogram)

    # Create figure with specified dimensions
    dpi = 100
    fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)

    # Create axes that fill the entire figure
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)

    # Display spectrogram with colormap
    librosa.display.specshow(
        spec_array,
        sr=SAMPLING_RATE,
        hop_length=256,
        cmap=cmap,
        ax=ax
    )
    fig.savefig(path, dpi=dpi, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    return spec_array

# Test the function
if 'spectrogram_dataset' in globals() and len(spectrogram_dataset['train']) > 0:
    test_path = "./test.png"
    print(np.array(spectrogram_dataset["train"][0]['audio']).shape)
    save_spectogram_to_file(spectrogram_dataset['train'][0]['audio'], test_path)
    print(f"Test image saved to: {test_path}")
else:
    print("spectrogram_dataset not available for testing")

(1025, 32)
Test image saved to: ./test.png


In [9]:
print("Saving spectrogram images to disk...")
for split in ['train', 'val', 'test']:
    split_data = spectrogram_dataset[split]
    for idx in tqdm(range(len(split_data)), desc=f"Processing {split} set"):
        item = split_data[idx]
        spectrogram = item['audio']
        label = LABELS[item['label']]

        # Define file path
        file_path = SPECTROGRAM_DIR / split / label / f"{idx:05d}_{split}.png"
        save_spectogram_to_file(spectrogram, file_path)

Saving spectrogram images to disk...


Processing test set: 100%|██████████| 434/434 [00:25<00:00, 16.74it/s]


In [10]:
# Verify spectrogram generation
print("\nVerifying spectrogram generation...")
for split in ['train', 'val', 'test']:
    split_path = SPECTROGRAM_DIR / split
    if split_path.exists():
        for label in LABELS:
            label_path = split_path / label
            if label_path.exists():
                num_files = len(list(label_path.glob("*.png")))
                print(f"{split}/{label}: {num_files} spectrograms")



Verifying spectrogram generation...
train/other: 1735 spectrograms
train/drone: 1786 spectrograms
val/other: 223 spectrograms
val/drone: 212 spectrograms
test/other: 224 spectrograms
test/drone: 210 spectrograms


In [11]:
# Create YOLO dataset configuration file
def create_yolo_config():
    """Create YOLO dataset configuration file."""
    config = {
        'path': str(SPECTROGRAM_DIR.absolute()),
        'train': 'train',
        'val': 'val',
        'test': 'test',
        'names': {0: 'other', 1: 'drone'},
        'nc': 2
    }

    config_path = SPECTROGRAM_DIR / 'dataset.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

    print(f"✓ Dataset config saved to {config_path}")
    return config_path

config_path = create_yolo_config()


✓ Dataset config saved to spectrograms/dataset.yaml


In [12]:
selected_size = "nano"
selected_version = "26"

YOLO_MODEL_SIZE = {
    "nano": "n",
    "small": "s",
    "medium": "m",
    "large": "l",
    "xlarge": "x",
}

# For classification, use the -cls suffix model name
# If model doesn't exist locally, YOLO will download it automatically
model_name = f"yolo{selected_version}{YOLO_MODEL_SIZE[selected_size]}-cls.pt"
model_path = Path('./models/') / model_name
model = YOLO(model_path, task="classify")

In [13]:
# Training configuration for classification
# Note: YOLO classification uses simpler parameters than detection
training_args = {
    'data': "./spectrograms/",  # Path to dataset directory with train/val/test subdirectories
    'epochs': 5,
    'imgsz': 480,  # Image size for classification (224 is standard)
    'batch': 32,
    'project': './runs/classify',
    'name': 'drone_audio_classification',
    'exist_ok': True,
    'pretrained': True,
    'optimizer': 'AdamW',
    'seed': 42,
    'deterministic': True,
    'cos_lr': True,
    'amp': True,
    'fraction': 1.0,
    'val': True,
    'task': 'classify',
}


In [15]:
# Train the model
print("Starting training...")
print("=" * 60)

results = model.train(**training_args)

print("\n" + "=" * 60)
print("Training completed!")
print(f"Results saved to: {results.save_dir}")


Starting training...


KeyError: 'model'

In [17]:
# Save the trained model
model_path = './drone_audio_yolo26_nano.pt'
model.export(format='onnx')  # Export to ONNX for deployment
model.save(model_path)
print(f"Model saved to: {model_path}")


Ultralytics 8.4.7 🚀 Python-3.13.5 torch-2.9.1+cu128 CPU (AMD EPYC 7513 32-Core Processor)
YOLO26n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs

PyTorch: starting from '/home/jovyan/runs/classify/runs/classify/drone_audio_classification/weights/best.pt' with input shape (1, 3, 480, 480) BCHW and output shape(s) (1, 2) (3.0 MB)

ONNX: starting export with onnx 1.20.1 opset 22...


[W126 15:12:34.910928699 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.911740281 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.911956103 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.912205974 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.912447415 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.912668405 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.912871706 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.913084627 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.913269367 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.913445088 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:1

ONNX: slimming with onnxslim 0.1.82...


[W126 15:12:34.139401218 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.141558846 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.144060765 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W126 15:12:34.146528154 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.


ONNX: export success ✅ 0.5s, saved as '/home/jovyan/runs/classify/runs/classify/drone_audio_classification/weights/best.onnx' (5.9 MB)

Export complete (0.7s)
Results saved to /home/jovyan/runs/classify/runs/classify/drone_audio_classification/weights
Predict:         yolo predict task=classify model=/home/jovyan/runs/classify/runs/classify/drone_audio_classification/weights/best.onnx imgsz=480 
Validate:        yolo val task=classify model=/home/jovyan/runs/classify/runs/classify/drone_audio_classification/weights/best.onnx imgsz=480 data=./spectrograms/  
Visualize:       https://netron.app
Model saved to: ./drone_audio_yolo26_nano.pt


In [20]:
!pip install seaborn -q

In [ ]:
# Comprehensive Metrics Evaluation on Train, Val, and Test Sets
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import seaborn as sns
from pathlib import Path
import pandas as pd

# Load the best model weights
best_model_path = results.save_dir / 'weights' / 'best.pt'
if not best_model_path.exists():
    # Try alternative path
    best_model_path = Path('./runs/classify/drone_audio_classification/weights/best.pt')
    
print(f"Loading model from: {best_model_path}")
best_model = YOLO(str(best_model_path), task='classify')

def evaluate_split(model, split_name, split_dir):
    """Evaluate model on a specific split and return predictions and true labels."""
    print(f"\n{'='*60}")
    print(f"Evaluating on {split_name.upper()} set")
    print(f"{'='*60}")
    
    # Get all images and their labels
    images = []
    true_labels = []
    
    for label_idx, label_name in enumerate(LABELS):
        label_dir = split_dir / label_name
        if label_dir.exists():
            image_files = list(label_dir.glob("*.png"))
            images.extend(image_files)
            true_labels.extend([label_idx] * len(image_files))
    
    print(f"Found {len(images)} images in {split_name} set")
    
    # Make predictions
    predictions = []
    predicted_probs = []
    
    for img_path in tqdm(images, desc=f"Predicting {split_name}"):
        results = model.predict(str(img_path), verbose=False)
        pred_class = int(results[0].probs.top1)
        pred_prob = float(results[0].probs.top1conf)
        predictions.append(pred_class)
        predicted_probs.append(pred_prob)
    
    return true_labels, predictions, predicted_probs

# Evaluate on all splits
splits_data = {}
for split_name in ['train', 'val', 'test']:
    split_dir = SPECTROGRAM_DIR / split_name
    true_labels, predictions, probs = evaluate_split(best_model, split_name, split_dir)
    splits_data[split_name] = {
        'true_labels': true_labels,
        'predictions': predictions,
        'probs': probs
    }

# Compute and display metrics for each split
print("\n" + "="*80)
print("COMPREHENSIVE METRICS SUMMARY")
print("="*80)

metrics_summary = []

for split_name in ['train', 'val', 'test']:
    true_labels = splits_data[split_name]['true_labels']
    predictions = splits_data[split_name]['predictions']
    
    # Calculate metrics
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels, predictions, average=None, zero_division=0
    )
    precision_macro = precision_recall_fscore_support(
        true_labels, predictions, average='macro', zero_division=0
    )[0]
    recall_macro = precision_recall_fscore_support(
        true_labels, predictions, average='macro', zero_division=0
    )[1]
    f1_macro = precision_recall_fscore_support(
        true_labels, predictions, average='macro', zero_division=0
    )[2]
    
    # Confusion matrix
    cm = confusion_matrix(true_labels, predictions)
    
    # Store metrics
    metrics_summary.append({
        'Split': split_name.upper(),
        'Accuracy': f"{accuracy:.4f}",
        'Precision (Macro)': f"{precision_macro:.4f}",
        'Recall (Macro)': f"{recall_macro:.4f}",
        'F1-Score (Macro)': f"{f1_macro:.4f}",
        'Samples': len(true_labels)
    })
    
    # Display detailed metrics
    print(f"\n{split_name.upper()} SET METRICS:")
    print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision (Macro): {precision_macro:.4f}")
    print(f"  Recall (Macro): {recall_macro:.4f}")
    print(f"  F1-Score (Macro): {f1_macro:.4f}")
    print(f"  Total Samples: {len(true_labels)}")
    
    print(f"\n  Per-Class Metrics:")
    for i, label_name in enumerate(LABELS):
        print(f"    {label_name}:")
        print(f"      Precision: {precision[i]:.4f}")
        print(f"      Recall: {recall[i]:.4f}")
        print(f"      F1-Score: {f1[i]:.4f}")
        print(f"      Support: {int(support[i])}")
    
    # Display confusion matrix
    print(f"\n  Confusion Matrix:")
    print(f"    {'':<10} {'Predicted:':<10} {'Predicted:'}")
    print(f"    {'':<10} {LABELS[0]:<10} {LABELS[1]}")
    for i, label_name in enumerate(LABELS):
        print(f"    True: {label_name:<6} {cm[i][0]:<10} {cm[i][1]}")
    
    # Classification report
    print(f"\n  Classification Report:")
    report = classification_report(true_labels, predictions, target_names=LABELS, zero_division=0)
    print(report)

# Create summary DataFrame
metrics_df = pd.DataFrame(metrics_summary)
print("\n" + "="*80)
print("METRICS SUMMARY TABLE")
print("="*80)
print(metrics_df.to_string(index=False))

# Visualize confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices for Train, Validation, and Test Sets', fontsize=16, y=1.02)

for idx, split_name in enumerate(['train', 'val', 'test']):
    true_labels = splits_data[split_name]['true_labels']
    predictions = splits_data[split_name]['predictions']
    cm = confusion_matrix(true_labels, predictions)

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=LABELS,
        yticklabels=LABELS,
        ax=axes[idx],
        cbar_kws={'label': 'Count'}
    )
    axes[idx].set_title(f'{split_name.upper()} Set\nAccuracy: {accuracy_score(true_labels, predictions):.4f}')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')

plt.tight_layout()
plt.show()

# Bar plot comparing metrics across splits
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Metrics Comparison Across Splits', fontsize=16, y=1.02)

splits = ['TRAIN', 'VAL', 'TEST']
accuracies = [float(metrics_summary[i]['Accuracy']) for i in range(3)]
precisions = [float(metrics_summary[i]['Precision (Macro)']) for i in range(3)]
recalls = [float(metrics_summary[i]['Recall (Macro)']) for i in range(3)]
f1_scores = [float(metrics_summary[i]['F1-Score (Macro)']) for i in range(3)]

axes[0, 0].bar(splits, accuracies, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0, 0].set_title('Accuracy')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(accuracies):
    axes[0, 0].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom')

axes[0, 1].bar(splits, precisions, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0, 1].set_title('Precision (Macro)')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_ylim([0, 1])
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(precisions):
    axes[0, 1].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom')

axes[1, 0].bar(splits, recalls, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Recall (Macro)')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_ylim([0, 1])
axes[1, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(recalls):
    axes[1, 0].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom')

axes[1, 1].bar(splits, f1_scores, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[1, 1].set_title('F1-Score (Macro)')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_ylim([0, 1])
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(f1_scores):
    axes[1, 1].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("Evaluation completed!")
print("="*80)


Loading model from: /home/jovyan/runs/classify/runs/classify/drone_audio_classification/weights/best.pt

Evaluating on TRAIN set
Found 352 images in train set


Predicting train: 100%|██████████| 352/352 [00:03<00:00, 110.73it/s]



Evaluating on VAL set
Found 43 images in val set


Predicting val: 100%|██████████| 43/43 [00:00<00:00, 112.55it/s]



Evaluating on TEST set
Found 43 images in test set


Predicting test: 100%|██████████| 43/43 [00:00<00:00, 110.02it/s]



COMPREHENSIVE METRICS SUMMARY

TRAIN SET METRICS:
  Accuracy: 0.1307 (13.07%)
  Precision (Macro): 0.1298
  Recall (Macro): 0.1310
  F1-Score (Macro): 0.1302
  Total Samples: 352

  Per-Class Metrics:
    other:
      Precision: 0.1145
      Recall: 0.1067
      F1-Score: 0.1105
      Support: 178
    drone:
      Precision: 0.1452
      Recall: 0.1552
      F1-Score: 0.1500
      Support: 174

  Confusion Matrix:
               Predicted: Predicted:
               other      drone
    True: other  19         159
    True: drone  147        27

  Classification Report:
              precision    recall  f1-score   support

       other       0.11      0.11      0.11       178
       drone       0.15      0.16      0.15       174

    accuracy                           0.13       352
   macro avg       0.13      0.13      0.13       352
weighted avg       0.13      0.13      0.13       352


VAL SET METRICS:
  Accuracy: 0.0698 (6.98%)
  Precision (Macro): 0.0683
  Recall (Macro): 0.078